In [ ]:
import pandas as pd
df = pd.read_excel('Wheat_modelling_data.xlsx')

In [ ]:
# Continuous variable used to DEFINE classes (clustered on train only)
y_cont = df['overall bread quality']

# Predictors (remove targets and any non-predictors)
# IMPORTANT: do NOT include Bread_Volume_2 itself as a predictor
drop_cols = ['YEAR', 'Growout', 'Sample ID', 'GIPSA Classification',
        'RVA-Peak time (min)',
       'RVA-Peak viscosity (RVU)', 'RVA-Breakdown (RVU)',
       'RVA-Final viscosity at 13 min (RVU)', 'L*', 'a*', 'b*', 'HMW-GS Composition',
       'Glu-A1', 'Glu-B1', 'Glu-D1',
       'Bread_Volume_2', 'overall bread quality', 'Crumb grain ',
       'crumb texture'] # add other targets if present
X_df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# Keep only rows with complete y_cont and predictors
mask = y_cont.notna() & X_df.notna().all(axis=1)
X_df = X_df.loc[mask].copy()
y_cont = y_cont.loc[mask].copy()

print(X_df.shape, y_cont.shape)


TOP K=10, 15 AND ALL

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.feature_selection import mutual_info_regression

# ============================================================
# 0) X, y  (assumes you already built X_df and y_cont)
# ============================================================
X0 = pd.get_dummies(X_df.copy(), drop_first=False).astype(float)
y = y_cont.copy()

# ============================================================
# 1) Train-only FS: Corr filter + (optional) MI top-k
# ============================================================
def corr_filter_train_only(X_train: pd.DataFrame, thresh: float = 0.85):
    corr = X_train.corr(numeric_only=True).abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [c for c in upper.columns if (upper[c] >= thresh).any()]
    X_red = X_train.drop(columns=to_drop, errors="ignore")
    return X_red, to_drop

def mi_rank_reg_train_only(X_train: pd.DataFrame, y_train: np.ndarray):
    mi = mutual_info_regression(X_train, y_train)
    mi_s = pd.Series(mi, index=X_train.columns).sort_values(ascending=False)
    return mi_s

def select_features_reg_train_only(
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    corr_thresh: float = 0.85,
    top_k: int | None = 10,   # None => use ALL after corr filter
):
    X_red, dropped = corr_filter_train_only(X_train, thresh=corr_thresh)
    kept_after_corr = X_red.columns.tolist()

    if top_k is None:
        feats = kept_after_corr
    else:
        mi_s = mi_rank_reg_train_only(X_red, y_train)
        k_use = min(int(top_k), mi_s.shape[0])
        feats = mi_s.index[:k_use].tolist()

    return feats, dropped, kept_after_corr

# ============================================================
# 2) CV runner for one "mode"
# ============================================================
def rf_regression_cv_one_mode(
    X: pd.DataFrame,
    y: pd.Series,
    n_splits: int = 5,
    random_state: int = 42,
    corr_thresh: float = 0.85,
    top_k: int | None = 10,
    rf_params: dict | None = None,
    mode_name: str = "topk10",
):
    if rf_params is None:
        rf_params = dict(
            n_estimators=1000,
            random_state=random_state,
            n_jobs=-1,
            min_samples_leaf=2,
        )

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    fold_rows = []
    fi_rows = []
    oof_pred = pd.Series(index=y.index, dtype=float)

    for fold, (tr_idx, te_idx) in enumerate(cv.split(X), start=1):
        X_train, X_test = X.iloc[tr_idx].copy(), X.iloc[te_idx].copy()
        y_train = y.iloc[tr_idx].to_numpy()
        y_test = y.iloc[te_idx].to_numpy()

        feats, dropped, kept_after_corr = select_features_reg_train_only(
            X_train, y_train, corr_thresh=corr_thresh, top_k=top_k
        )

        model = RandomForestRegressor(**rf_params)
        model.fit(X_train[feats], y_train)

        pred = model.predict(X_test[feats])
        oof_pred.iloc[te_idx] = pred

        mae = mean_absolute_error(y_test, pred)
        mse = mean_squared_error(y_test, pred)
        rmse = float(np.sqrt(mse))
        r2 = r2_score(y_test, pred)

        fold_rows.append({
            "mode": mode_name,
            "fold": fold,
            "n_train": len(tr_idx),
            "n_test": len(te_idx),
            "n_feats_used": len(feats),
            "n_feats_after_corr": len(kept_after_corr),
            "dropped_corr_n": len(dropped),
            "MAE": float(mae),
            "MSE": float(mse),
            "RMSE": float(rmse),
            "R2": float(r2),
        })

        fi = pd.Series(model.feature_importances_, index=feats).sort_values(ascending=False)
        for f, v in fi.items():
            fi_rows.append({"mode": mode_name, "fold": fold, "feature": f, "importance": float(v)})

    fold_df = pd.DataFrame(fold_rows)

    metrics = ["MAE", "MSE", "RMSE", "R2"]
    cv_summary = (fold_df.groupby("mode")[metrics]
                  .agg(["mean", "std"])
                  .reset_index())
    cv_summary.columns = (
        ["mode"]
        + [f"{m}_{stat}" for m in metrics for stat in ["mean", "std"]]
    )

    y_true = y.loc[oof_pred.index].to_numpy()
    y_hat = oof_pred.to_numpy()
    oof_summary = pd.DataFrame([{
        "mode": mode_name,
        "OOF_MAE": float(mean_absolute_error(y_true, y_hat)),
        "OOF_MSE": float(mean_squared_error(y_true, y_hat)),
        "OOF_RMSE": float(np.sqrt(mean_squared_error(y_true, y_hat))),
        "OOF_R2": float(r2_score(y_true, y_hat)),
    }])

    fi_long = pd.DataFrame(fi_rows)
    fi_summary = (fi_long.groupby(["mode", "feature"], as_index=False)["importance"]
                  .agg(mean_importance="mean", sd_importance="std")
                  .sort_values(["mode", "mean_importance"], ascending=[True, False]))
    fi_summary["sd_importance"] = fi_summary["sd_importance"].fillna(0.0)

    return fold_df, cv_summary, oof_summary, fi_summary, fi_long, oof_pred


# ============================================================
# 3) Run multiple modes in ONE go: top10, top15, all
# ============================================================
def rf_regression_cv_multi_modes(
    X: pd.DataFrame,
    y: pd.Series,
    modes: list[tuple[str, int | None]],  # e.g. [("topk10",10),("topk15",15),("all",None)]
    n_splits: int = 5,
    random_state: int = 42,
    corr_thresh: float = 0.85,
    rf_params: dict | None = None,
):
    fold_dfs, cv_summaries, oof_summaries = [], [], []
    fi_summaries, fi_longs = [], []
    oof_preds = {}

    for mode_name, top_k in modes:
        fold_df, cv_summary, oof_summary, fi_summary, fi_long, oof_pred = rf_regression_cv_one_mode(
            X=X, y=y,
            n_splits=n_splits, random_state=random_state,
            corr_thresh=corr_thresh, top_k=top_k,
            rf_params=rf_params,
            mode_name=mode_name
        )
        fold_dfs.append(fold_df)
        cv_summaries.append(cv_summary)
        oof_summaries.append(oof_summary)
        fi_summaries.append(fi_summary)
        fi_longs.append(fi_long)
        oof_preds[mode_name] = oof_pred

    return (
        pd.concat(fold_dfs, ignore_index=True),
        pd.concat(cv_summaries, ignore_index=True),
        pd.concat(oof_summaries, ignore_index=True),
        pd.concat(fi_summaries, ignore_index=True),
        pd.concat(fi_longs, ignore_index=True),
        oof_preds
    )


# ======================
# RUN
# ======================
rf_params = dict(
    n_estimators=1000,
    random_state=42,
    n_jobs=-1,
    min_samples_leaf=2,
)

modes = [("topk10", 10), ("topk15", 15), ("all", None)]

fold_df, cv_summary, oof_summary, fi_summary, fi_long, oof_preds = rf_regression_cv_multi_modes(
    X=X0, y=y,
    modes=modes,
    n_splits=5,
    random_state=42,
    corr_thresh=0.8,
    rf_params=rf_params
)

print("\n=== Fold-wise regression metrics (top10, top15, all) ===")
print(fold_df)

print("\n=== CV mean ± SD (MAE/MSE/RMSE/R2) ===")
print(cv_summary)

print("\n=== Overall OOF regression metrics ===")
print(oof_summary)

print("\n=== Top 20 FI per mode (mean importance) ===")
for mode_name, _ in modes:
    print(f"\n--- {mode_name} ---")
    print(fi_summary[fi_summary["mode"] == mode_name].head(20))

# Optional saves
fold_df.to_csv("rf_regression_fold_metrics_top10_top15_all.csv", index=False) 
cv_summary.to_csv("rf_regression_cv_mean_sd_top10_top15_all.csv", index=False)
# oof_summary.to_csv("rf_regression_oof_metrics_top10_top15_all.csv", index=False)
# fi_summary.to_csv("rf_regression_fi_mean_sd_top10_top15_all.csv", index=False)
# pd.DataFrame({k: v for k, v in oof_preds.items()}).to_csv("rf_regression_oof_predictions_by_mode.csv", index=False)

VALIDATION

In [ ]:
import pandas as pd
import numpy as np

# validation data
val = pd.read_excel(
    'Wheat-validation-data.xlsx'
)

# Continuous variable for validation
y_val_cont = val['overall bread quality'].copy()

drop_cols = ['YEAR', 'Growout', 'Sample ID', 'GIPSA Classification',
        'RVA-Peak time (min)',
       'RVA-Peak viscosity (RVU)', 'RVA-Breakdown (RVU)',
       'RVA-Final viscosity at 13 min (RVU)', 'L*', 'a*', 'b*', 'HMW-GS Composition',
       'Glu-A1', 'Glu-B1', 'Glu-D1',
       'Bread_Volume_2', 'overall bread quality', 'Crumb grain ',
       'crumb texture']

# IMPORTANT: use val.columns (not df.columns)
X_val_df = val.drop(columns=[c for c in drop_cols if c in val.columns])

# keep complete rows (on val set)
mask = y_val_cont.notna() & X_val_df.notna().all(axis=1)
X_val_df = X_val_df.loc[mask].copy()
y_val_cont = y_val_cont.loc[mask].copy()

print("X_val_df:", X_val_df.shape, "y_val_cont:", y_val_cont.shape)
# ============================================================
# 4) Prepare TRAIN matrix and VAL matrix with identical columns
# ============================================================
X_train0 = pd.get_dummies(X_df.copy(), drop_first=False).astype(float)

X_val0 = pd.get_dummies(X_val_df.copy(), drop_first=False).astype(float)

# Align validation columns to training columns:
# - adds missing dummy columns in val as 0
# - drops extra columns not seen in training
X_val0 = X_val0.reindex(columns=X_train0.columns, fill_value=0.0)

print("X_train0:", X_train0.shape)
print("X_val0  :", X_val0.shape)


In [ ]:
# ============================================================
# 5) Train FINAL RF on FULL training and predict validation
#     for topk10, topk15, all
# ============================================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ensure y is aligned with X_train0
y_train_full = y_cont.loc[X_train0.index].copy()

rf_params = dict(
    n_estimators=800,
    random_state=42,
    n_jobs=-1,
    min_samples_leaf=2,
)

modes = [("topk10", 10), ("topk15", 15), ("all", None)]

predictions_val = {}
metrics_val = []

for mode_name, top_k in modes:
    # ---- feature selection FIT on FULL TRAIN ----
    feats, dropped, kept_after_corr = select_features_reg_train_only(
        X_train0, y_train_full.to_numpy(),
        corr_thresh=0.80,
        top_k=top_k
    )

    # ---- train final RF on FULL TRAIN ----
    rf_final = RandomForestRegressor(**rf_params)
    rf_final.fit(X_train0[feats], y_train_full)

    # ---- predict external validation ----
    y_val_pred = rf_final.predict(X_val0[feats])
    predictions_val[mode_name] = y_val_pred

    # ---- validation metrics (optional, but recommended) ----
    mae = mean_absolute_error(y_val_cont, y_val_pred)
    rmse = np.sqrt(mean_squared_error(y_val_cont, y_val_pred))
    r2 = r2_score(y_val_cont, y_val_pred)

    metrics_val.append({
        "mode": mode_name,
        "n_feats_used": len(feats),
        "MAE_val": float(mae),
        "RMSE_val": float(rmse),
        "R2_val": float(r2),
    })

val_metrics_df = pd.DataFrame(metrics_val)
print("\n=== External validation metrics (final RF trained on full train) ===")
print(val_metrics_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(f"{val_metrics_df.to_string(index=False)}")
val_metrics_df.to_csv("RF_validation_metrics.csv", index=False)

In [ ]:
# ============================================================
# 6) Observed vs Predicted table (ALL rows)
# ============================================================

comparison_df = pd.DataFrame({
    "Observed": np.asarray(y_val_cont),
    "RF_topk10": predictions_val["topk10"],
    "RF_topk15": predictions_val["topk15"],
    "RF_all": predictions_val["all"],
}).reset_index(drop=True)

print("\n=== Observed vs Predicted (External Validation) ===")
print(comparison_df.round(2).to_string(index=False))
print("Rows:", len(comparison_df))

# Save as Excel
comparison_df.to_excel("OBQ-RF_external_validation_observed_vs_predicted.xlsx", index=False)

print("Saved: OBQ-RF_external_validation_observed_vs_predicted.xlsx")
